In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: True
GPU: Tesla T4


In [6]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/quester11
/kaggle/input/datasets/quester11/cig-dataset
/kaggle/input/datasets/quester11/cig-dataset/train_images
/kaggle/input/datasets/quester11/cig-dataset/train_images/train_images
/kaggle/input/datasets/quester11/cig-dataset/test_images
/kaggle/input/datasets/quester11/cig-dataset/test_images/test_images


In [10]:
import os

print(os.listdir('/kaggle/input'))

['datasets']


In [11]:
!pip install -q albumentations jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 35.9 MB/s eta 0:00:0000:0100:01


In [12]:
import os
import cv2
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

import albumentations as A

from jiwer import cer

In [13]:
CSV_PATH = "/kaggle/input/datasets/quester11/cig-dataset/train-labels.csv"

df = pd.read_csv(CSV_PATH)

print(df.head())
print(df.shape)

   Unnamed: 0        image    text
0           0  train-0.png  BU522X
1           1  train-1.png  XQ8NE2
2           2  train-2.png  DTZD3E
3           3  train-3.png  SM424H
4           4  train-4.png  6YVTQR
(20000, 3)


In [23]:
df.loc[df["image"]=="train-2184.png","text"] = "5396E9"
df.loc[df["image"]=="train-6819.png","text"] = "4MARS4"

In [24]:
print(df[df["image"].isin(["train-2184.png","train-6819.png"])])

      Unnamed: 0           image    text
2184        2184  train-2184.png  5396E9
6819        6819  train-6819.png  4MARS4


In [25]:
charset = sorted(set("".join(df["text"])))

print(charset)
print("Classes:", len(charset))

char2idx = {c:i for i,c in enumerate(charset)}
idx2char = {i:c for c,i in char2idx.items()}

['2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'J', 'K', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
Classes: 31


In [26]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print(len(train_df))
print(len(val_df))

16000
4000


In [27]:
train_tfms = A.Compose([
    A.GaussNoise(p=0.4),

    A.MotionBlur(
        blur_limit=(3,5),
        p=0.3
    ),

    A.RandomBrightnessContrast(
        p=0.4
    ),

    A.GridDistortion(
        p=0.3
    ),

    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.1,
        rotate_limit=10,
        p=0.5
    ),
])

val_tfms = A.Compose([])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [28]:
class CaptchaDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_dir,
        transforms=None
    ):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        image_path = os.path.join(
            self.image_dir,
            row["image"]
        )

        img = cv2.imread(
            image_path,
            cv2.IMREAD_GRAYSCALE
        )

        if self.transforms:
            img = self.transforms(image=img)["image"]

        img = img.astype(np.float32) / 255.0

        img = np.expand_dims(img, axis=0)

        label = [
            char2idx[c]
            for c in row["text"]
        ]

        return (
            torch.tensor(img, dtype=torch.float32),
            torch.tensor(label, dtype=torch.long)
        )

In [29]:
TRAIN_DIR = "/kaggle/input/datasets/quester11/cig-dataset/train_images/train_images"

train_ds = CaptchaDataset(
    train_df,
    TRAIN_DIR,
    train_tfms
)

val_ds = CaptchaDataset(
    val_df,
    TRAIN_DIR,
    val_tfms
)

train_loader = DataLoader(
    train_ds,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [30]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

print(labels[0])

torch.Size([64, 1, 100, 200])
torch.Size([64, 6])
tensor([15, 12, 18, 14, 30, 20])


In [31]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),

            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):

        b, c, _, _ = x.size()

        y = self.pool(x).view(b, c)

        y = self.fc(y).view(b, c, 1, 1)

        return x * y

In [32]:
class ResidualBlock(nn.Module):

    def __init__(self, in_ch, out_ch):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_ch,
                out_ch,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_ch),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_ch,
                out_ch,
                3,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(out_ch)
        )

        self.shortcut = nn.Sequential()

        if in_ch != out_ch:

            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_ch,
                    out_ch,
                    1,
                    bias=False
                ),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):

        out = self.conv(x)

        out += self.shortcut(x)

        out = torch.relu(out)

        return out

In [33]:
class CaptchaNet(nn.Module):

    def __init__(self, num_classes=31):

        super().__init__()

        self.stem = nn.Sequential(

            nn.Conv2d(
                1,
                64,
                3,
                padding=1
            ),

            nn.BatchNorm2d(64),

            nn.ReLU(inplace=True),

            nn.MaxPool2d(2)
        )

        self.block1 = ResidualBlock(64, 64)

        self.block2 = ResidualBlock(64, 128)

        self.se1 = SEBlock(128)

        self.pool1 = nn.MaxPool2d(2)

        self.block3 = ResidualBlock(128, 256)

        self.block4 = ResidualBlock(256, 512)

        self.se2 = SEBlock(512)

        self.pool2 = nn.AdaptiveAvgPool2d(1)

        self.dropout = nn.Dropout(0.3)

        self.heads = nn.ModuleList(
            [
                nn.Linear(512, num_classes)
                for _ in range(6)
            ]
        )

    def forward(self, x):

        x = self.stem(x)

        x = self.block1(x)

        x = self.block2(x)

        x = self.se1(x)

        x = self.pool1(x)

        x = self.block3(x)

        x = self.block4(x)

        x = self.se2(x)

        x = self.pool2(x)

        x = x.flatten(1)

        x = self.dropout(x)

        outputs = []

        for head in self.heads:
            outputs.append(head(x))

        return outputs

In [34]:
device = "cuda"

model = CaptchaNet(
    num_classes=len(charset)
).to(device)

sample = images[:4].to(device)

out = model(sample)

print(len(out))

for x in out:
    print(x.shape)

6
torch.Size([4, 31])
torch.Size([4, 31])
torch.Size([4, 31])
torch.Size([4, 31])
torch.Size([4, 31])
torch.Size([4, 31])


In [35]:
criterion = nn.CrossEntropyLoss()

In [36]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [37]:
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30
)

In [38]:
def decode_labels(label_tensor):
    
    result = []

    for row in label_tensor:
        
        text = "".join(
            idx2char[int(x)]
            for x in row
        )

        result.append(text)

    return result


def decode_predictions(outputs):

    preds = []

    for head in outputs:
        preds.append(
            torch.argmax(head, dim=1)
        )

    preds = torch.stack(preds, dim=1)

    return decode_labels(
        preds.cpu()
    )

In [39]:
def evaluate(model, loader):

    model.eval()

    total_cer = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            pred_texts = decode_predictions(outputs)

            true_texts = decode_labels(labels)

            for p, t in zip(
                pred_texts,
                true_texts
            ):

                total_cer += cer(t, p)
                total += 1

    return total_cer / total

In [40]:
best_cer = 999

epochs = 30

for epoch in range(epochs):

    model.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = 0

        for i in range(6):

            loss += criterion(
                outputs[i],
                labels[:, i]
            )

        loss /= 6

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    val_cer = evaluate(
        model,
        val_loader
    )

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss: {running_loss/len(train_loader):.4f} | "
        f"CER: {val_cer:.4f}"
    )

    if val_cer < best_cer:

        best_cer = val_cer

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

        print(
            f"Saved model with CER {best_cer:.4f}"
        )

Epoch 1/30 | Loss: 3.4206 | CER: 0.9457
Saved model with CER 0.9457
Epoch 2/30 | Loss: 3.3464 | CER: 0.9564
Epoch 3/30 | Loss: 3.2171 | CER: 0.9068
Saved model with CER 0.9068
Epoch 4/30 | Loss: 2.9145 | CER: 0.7901
Saved model with CER 0.7901
Epoch 5/30 | Loss: 2.4506 | CER: 0.7055
Saved model with CER 0.7055
Epoch 6/30 | Loss: 2.0043 | CER: 0.5255
Saved model with CER 0.5255
Epoch 7/30 | Loss: 1.5996 | CER: 0.4616
Saved model with CER 0.4616
Epoch 8/30 | Loss: 1.3933 | CER: 0.4515
Saved model with CER 0.4515
Epoch 9/30 | Loss: 1.3024 | CER: 0.4502
Saved model with CER 0.4502
Epoch 10/30 | Loss: 1.2467 | CER: 0.4458
Saved model with CER 0.4458
Epoch 11/30 | Loss: 1.2153 | CER: 0.4431
Saved model with CER 0.4431
Epoch 12/30 | Loss: 1.1916 | CER: 0.4460
Epoch 13/30 | Loss: 1.1719 | CER: 0.4382
Saved model with CER 0.4382
Epoch 14/30 | Loss: 1.1524 | CER: 0.4448
Epoch 15/30 | Loss: 1.1387 | CER: 0.4353
Saved model with CER 0.4353
Epoch 16/30 | Loss: 1.1236 | CER: 0.4280
Saved model with 

In [41]:
images, labels = next(iter(val_loader))

images = images.to(device)

with torch.no_grad():
    outputs = model(images)

pred_texts = decode_predictions(outputs)
true_texts = decode_labels(labels)

for p, t in list(zip(pred_texts, true_texts))[:20]:
    print("Pred:", p, " | True:", t)

Pred: 48TTDN  | True: 48TADN
Pred: 9GEGA6  | True: 9JEGA6
Pred: 39FYFC  | True: 3JY9FC
Pred: GXNNGU  | True: GXNHGU
Pred: B4343J  | True: B4238J
Pred: ETTT57  | True: E5TJ97
Pred: F5QQKZ  | True: F5QPKZ
Pred: 4GFFCD  | True: 4GRFCD
Pred: 9VQVV6  | True: 95JQV6
Pred: HADMDW  | True: H3DMAW
Pred: U48BHM  | True: U48BHM
Pred: BESS7W  | True: BESG7W
Pred: MTTT8S  | True: MT2W8S
Pred: DMMAAM  | True: DKMPAM
Pred: J6666F  | True: J6U76F
Pred: MTTTT8  | True: MT3TS8
Pred: M9Y22F  | True: M9YU2F
Pred: WSYY58  | True: WR5YS8
Pred: ZRRRRH  | True: ZBR6RH
Pred: YGZZZX  | True: YGZXZX


In [42]:
# ==========================================
# VERSION 2 MODEL (CRNN + ATTENTION)
# ==========================================

In [43]:
torch.save(
    model.state_dict(),
    "/kaggle/working/baseline_model.pth"
)

print("Baseline saved")

Baseline saved


In [44]:
class ChannelAttention(nn.Module):

    def __init__(self, channels, reduction=16):
        super().__init__()

        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.fc = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(channels // reduction, channels, 1, bias=False)
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg = self.fc(self.avg_pool(x))
        mx = self.fc(self.max_pool(x))

        return self.sigmoid(avg + mx)

In [45]:
class SpatialAttention(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size=7,
            padding=3,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg = torch.mean(
            x,
            dim=1,
            keepdim=True
        )

        mx, _ = torch.max(
            x,
            dim=1,
            keepdim=True
        )

        x = torch.cat([avg, mx], dim=1)

        return self.sigmoid(self.conv(x))

In [46]:
class CBAM(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.ca = ChannelAttention(channels)
        self.sa = SpatialAttention()

    def forward(self, x):

        x = x * self.ca(x)
        x = x * self.sa(x)

        return x

In [47]:
class ResidualBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()

        if stride != 1 or in_channels != out_channels:

            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):

        out = torch.relu(
            self.bn1(
                self.conv1(x)
            )
        )

        out = self.bn2(
            self.conv2(out)
        )

        out += self.shortcut(x)

        return torch.relu(out)

In [48]:
class CNNBackbone(nn.Module):

    def __init__(self):
        super().__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(
                1,
                64,
                3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.layer2 = ResidualBlock(
            64,
            128,
            stride=2
        )

        self.cbam1 = CBAM(128)

        self.layer3 = ResidualBlock(
            128,
            256,
            stride=2
        )

        self.cbam2 = CBAM(256)

        self.layer4 = ResidualBlock(
            256,
            512,
            stride=2
        )

    def forward(self, x):

        x = self.layer1(x)

        x = self.layer2(x)

        x = self.cbam1(x)

        x = self.layer3(x)

        x = self.cbam2(x)

        x = self.layer4(x)

        return x

In [49]:
backbone = CNNBackbone().cuda()

dummy = torch.randn(
    2,
    1,
    100,
    200
).cuda()

out = backbone(dummy)

print(out.shape)

torch.Size([2, 512, 7, 13])


In [50]:
class SequenceEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.height_pool = nn.AdaptiveAvgPool2d(
            (1, None)
        )

    def forward(self, x):

        x = self.height_pool(x)

        x = x.squeeze(2)

        x = x.permute(
            0,
            2,
            1
        )

        return x

In [51]:
encoder = SequenceEncoder().cuda()

seq = encoder(out)

print(seq.shape)

torch.Size([2, 13, 512])


In [52]:
class BiLSTM(nn.Module):

    def __init__(
        self,
        input_size=512,
        hidden_size=256,
        num_layers=2
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.2
        )

    def forward(self, x):

        x, _ = self.lstm(x)

        return x

In [53]:
lstm = BiLSTM().cuda()

lstm_out = lstm(seq)

print(lstm_out.shape)

torch.Size([2, 13, 512])


In [54]:
class SelfAttention(nn.Module):

    def __init__(
        self,
        embed_dim=512,
        num_heads=8
    ):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim,
            num_heads,
            batch_first=True
        )

    def forward(self, x):

        x, _ = self.attn(
            x,
            x,
            x
        )

        return x

In [55]:
attn = SelfAttention().cuda()

attn_out = attn(lstm_out)

print(attn_out.shape)

torch.Size([2, 13, 512])


In [56]:
class CharacterDecoder(nn.Module):

    def __init__(
        self,
        embed_dim=512,
        num_classes=31
    ):
        super().__init__()

        self.pool = nn.AdaptiveAvgPool1d(6)

        self.head = nn.Linear(
            embed_dim,
            num_classes
        )

    def forward(self, x):

        x = x.permute(
            0,
            2,
            1
        )

        x = self.pool(x)

        x = x.permute(
            0,
            2,
            1
        )

        return self.head(x)

In [57]:
decoder = CharacterDecoder(
    num_classes=len(charset)
).cuda()

decoded = decoder(attn_out)

print(decoded.shape)

torch.Size([2, 6, 31])


In [58]:
class CRNNAttention(nn.Module):

    def __init__(self, num_classes=31):
        super().__init__()

        self.backbone = CNNBackbone()

        self.encoder = SequenceEncoder()

        self.lstm = BiLSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2
        )

        self.attention = SelfAttention(
            embed_dim=512,
            num_heads=8
        )

        self.decoder = CharacterDecoder(
            embed_dim=512,
            num_classes=num_classes
        )

    def forward(self, x):

        x = self.backbone(x)

        x = self.encoder(x)

        x = self.lstm(x)

        x = self.attention(x)

        x = self.decoder(x)

        return x

In [59]:
device = "cuda"

model_v2 = CRNNAttention(
    num_classes=len(charset)
).to(device)

dummy = torch.randn(
    4,
    1,
    100,
    200
).to(device)

out = model_v2(dummy)

print(out.shape)

torch.Size([4, 6, 31])


In [60]:
criterion = nn.CrossEntropyLoss()

In [61]:
optimizer = optim.AdamW(
    model_v2.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

In [62]:
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=40
)

In [63]:
def decode_predictions_v2(logits):

    preds = torch.argmax(
        logits,
        dim=2
    )

    result = []

    for row in preds:

        text = "".join(
            idx2char[int(x)]
            for x in row
        )

        result.append(text)

    return result

In [64]:
def evaluate_v2(model, loader):

    model.eval()

    total_cer = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            logits = model(images)

            pred_texts = decode_predictions_v2(
                logits.cpu()
            )

            true_texts = decode_labels(
                labels
            )

            for p, t in zip(
                pred_texts,
                true_texts
            ):

                total_cer += cer(
                    t,
                    p
                )

                total += 1

    return total_cer / total

In [66]:
best_cer = 999

epochs = 40

for epoch in range(epochs):

    model_v2.train()

    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model_v2(images)

        loss = 0

        for pos in range(6):

            loss += criterion(
                logits[:, pos, :],
                labels[:, pos]
            )

        loss /= 6

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model_v2.parameters(),
            5.0
        )

        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    val_cer = evaluate_v2(
        model_v2,
        val_loader
    )

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Loss: {running_loss/len(train_loader):.4f} | "
        f"CER: {val_cer:.4f}"
    )

    if val_cer < best_cer:

        best_cer = val_cer

        torch.save(
            model_v2.state_dict(),
            "best_crnn_attention.pth"
        )

        print(
            f"Saved Best CER = {best_cer:.4f}"
        )

KeyboardInterrupt: 

In [67]:
model_v2.load_state_dict(
    torch.load("best_crnn_attention.pth")
)

model_v2.eval()

print("Best model loaded")

Best model loaded


In [68]:
images, labels = next(iter(val_loader))

images = images.to(device)

with torch.no_grad():
    logits = model_v2(images)

pred_texts = decode_predictions_v2(
    logits.cpu()
)

true_texts = decode_labels(labels)

for i in range(20):
    print(
        pred_texts[i],
        "|",
        true_texts[i]
    )

48TADN | 48TADN
9JEGA6 | 9JEGA6
3JY9FC | 3JY9FC
GXNHGU | GXNHGU
B4238J | B4238J
E5TJ97 | E5TJ97
F5QPKZ | F5QPKZ
4GRFCD | 4GRFCD
95JQV6 | 95JQV6
H3DMAW | H3DMAW
U48BHM | U48BHM
BESG7W | BESG7W
MT2W8S | MT2W8S
DKMPAM | DKMPAM
J6U76F | J6U76F
MT3TS8 | MT3TS8
M9YU2F | M9YU2F
WR5YS8 | WR5YS8
ZBR6RH | ZBR6RH
YGZXZX | YGZXZX


In [69]:
import os

TEST_DIR = "/kaggle/input/datasets/quester11/cig-dataset/test_images/test_images"

files = sorted(os.listdir(TEST_DIR))

print(len(files))
print(files[:10])

5000
['test-0.png', 'test-1.png', 'test-10.png', 'test-100.png', 'test-1000.png', 'test-1001.png', 'test-1002.png', 'test-1003.png', 'test-1004.png', 'test-1005.png']


In [73]:
class TestDataset(Dataset):

    def __init__(self, image_dir):

        self.image_dir = image_dir

        self.files = sorted(
            os.listdir(image_dir),
            key=lambda x: int(
                x.replace("test-", "")
                 .replace(".png", "")
            )
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        filename = self.files[idx]

        path = os.path.join(
            self.image_dir,
            filename
        )

        img = cv2.imread(
            path,
            cv2.IMREAD_GRAYSCALE
        )

        img = img.astype(np.float32) / 255.0

        img = np.expand_dims(
            img,
            axis=0
        )

        return (
            torch.tensor(
                img,
                dtype=torch.float32
            ),
            filename
        )

In [74]:
TEST_DIR = "/kaggle/input/datasets/quester11/cig-dataset/test_images/test_images"

test_ds = TestDataset(TEST_DIR)

test_loader = DataLoader(
    test_ds,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

print(len(test_ds))

5000


In [75]:
model_v2.load_state_dict(
    torch.load(
        "best_crnn_attention.pth"
    )
)

model_v2.eval()

print("Best model loaded")

Best model loaded


In [76]:
predictions = []

with torch.no_grad():

    for images, filenames in test_loader:

        images = images.to(device)

        logits = model_v2(images)

        pred_texts = decode_predictions_v2(
            logits.cpu()
        )

        for fname, text in zip(
            filenames,
            pred_texts
        ):
            predictions.append(
                [fname, text]
            )

print(len(predictions))
print(predictions[:5])

5000
[['test-0.png', 'QVTQ8A'], ['test-1.png', '7PSW9D'], ['test-2.png', 'WJ2WNY'], ['test-3.png', 'RFHJD4'], ['test-4.png', 'K7ZUF2']]


In [77]:
submission = pd.DataFrame(
    predictions,
    columns=[
        "image",
        "text"
    ]
)

submission = submission.sort_values(
    "image",
    key=lambda s: s.str.extract(r'(\d+)')[0].astype(int)
)

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,image,text
0,test-0.png,QVTQ8A
1,test-1.png,7PSW9D
2,test-2.png,WJ2WNY
3,test-3.png,RFHJD4
4,test-4.png,K7ZUF2


In [78]:
print(submission.shape)

print(submission.head())

print(submission.tail())

(5000, 2)
        image    text
0  test-0.png  QVTQ8A
1  test-1.png  7PSW9D
2  test-2.png  WJ2WNY
3  test-3.png  RFHJD4
4  test-4.png  K7ZUF2
              image    text
4995  test-4995.png  R5ENAF
4996  test-4996.png  WNQH33
4997  test-4997.png  9SYANQ
4998  test-4998.png  9X7DDR
4999  test-4999.png  Z2WTZ3


In [79]:
submission = pd.DataFrame(
    predictions,
    columns=[
        "image",
        "prediction"
    ]
)

submission = submission.sort_values(
    "image",
    key=lambda s: s.str.extract(r'(\d+)')[0].astype(int)
)

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,image,prediction
0,test-0.png,QVTQ8A
1,test-1.png,7PSW9D
2,test-2.png,WJ2WNY
3,test-3.png,RFHJD4
4,test-4.png,K7ZUF2


In [80]:
print(submission.shape)

print(submission.head())

print(submission.tail())

(5000, 2)
        image prediction
0  test-0.png     QVTQ8A
1  test-1.png     7PSW9D
2  test-2.png     WJ2WNY
3  test-3.png     RFHJD4
4  test-4.png     K7ZUF2
              image prediction
4995  test-4995.png     R5ENAF
4996  test-4996.png     WNQH33
4997  test-4997.png     9SYANQ
4998  test-4998.png     9X7DDR
4999  test-4999.png     Z2WTZ3


In [81]:
model_v2.load_state_dict(
    torch.load("best_crnn_attention.pth")
)

model_v2.eval()

best_cer = evaluate_v2(model_v2, val_loader)

char_accuracy = (1 - best_cer) * 100

print("="*60)
print("FINAL MODEL EVALUATION")
print("="*60)
print(f"Validation CER       : {best_cer:.6f}")
print(f"Character Accuracy   : {char_accuracy:.4f}%")
print("="*60)

FINAL MODEL EVALUATION
Validation CER       : 0.000125
Character Accuracy   : 99.9875%
